# 00 · Preparar o Drive — **rodar em casa, uma vez só**

Este notebook não é de palco. Ele cria a estrutura de pastas no seu Drive,
baixa os modelos e deixa tudo pronto para as demos rodarem **sem depender de
download na hora da palestra**.

Raiz de tudo: `/content/drive/MyDrive/PALESTRA-IA`

Depois de rodar, você vai preencher três pastas com imagens:

| pasta | o que colocar | quantas |
|---|---|---|
| `04-garrafas/inferencia/` | fotos de bancada/prateleira **com várias garrafas** | 5–10 |
| `04-garrafas/treino/train/lacrada` e `/aberta` | garrafas **isoladas**, uma por foto | 60+ de cada |
| `05-epi/treino/train/com_epi` e `/sem_epi` | pessoas **com** e **sem** capacete/óculos | 80+ de cada |

E o mesmo, com **20% do volume**, nas pastas `val/` — são as fotos que o modelo
**nunca vê no treino** e que provam que ele aprendeu de verdade, em vez de ter
decorado. Sem isso, o número que aparece na tela não vale nada.

In [ ]:
# ── 1. instala a biblioteca e monta o Google Drive ──
%pip install -q ultralytics
from google.colab import drive
drive.mount('/content/drive')

from ultralytics import YOLO
import ultralytics, torch, os, glob
ultralytics.checks()
print("GPU disponivel:", torch.cuda.is_available())

# ── 2. a raiz de tudo, e a conferência de que ela é REAL ──────────────
#
# ARMADILHA que já custou uma sessão: a linha acima cria a variável
# `drive` (minúscula), que é o MÓDULO do Colab. Se algum caminho for
# escrito com `drive` em vez de `DRIVE`, o Python aceita numa boa e
# monta um caminho como
#     <module 'google.colab.drive' from '/usr/local/...'>/04-garrafas
# O código roda, cria pastas, exporta arquivos — tudo no disco
# temporário do Colab, que evapora quando a sessão encerra. Nada disso
# chega ao seu Drive, e não há erro nenhum na tela.
#
# A conferência abaixo transforma esse silêncio num aviso imediato.

DRIVE = "/content/drive/MyDrive/PALESTRA-IA"

if not DRIVE.startswith("/content/drive/"):
    raise SystemExit(
        "DRIVE aponta para fora do Google Drive: " + repr(DRIVE) + "\n"
        "Provavelmente algum caminho usou `drive` (o módulo) em vez de `DRIVE`.")
if not os.path.isdir("/content/drive/MyDrive"):
    raise SystemExit("O Drive não montou. Rode esta célula de novo e autorize o acesso.")

os.makedirs(DRIVE, exist_ok=True)
print("raiz no Drive:", DRIVE)
print("existe de verdade:", os.path.isdir(DRIVE))

In [ ]:
# ── 2. cria a arvore de pastas ──
import os
PASTAS = [
    "00-pesos",
    "01-deteccao/entrada",
    "01-deteccao/saida",
    "02-segmentacao/entrada",
    "02-segmentacao/saida",
    "03-pose/entrada",
    "03-pose/saida",
    "04-garrafas/inferencia",
    "04-garrafas/saida",
    "04-garrafas/pesos",
    "04-garrafas/treino/train/lacrada",
    "04-garrafas/treino/train/aberta",
    "04-garrafas/treino/val/lacrada",
    "04-garrafas/treino/val/aberta",
    "05-epi/saida",
    "05-epi/pesos",
    "05-epi/treino/train/com_epi",
    "05-epi/treino/train/sem_epi",
    "05-epi/treino/val/com_epi",
    "05-epi/treino/val/sem_epi",
    "99-reserva"
]
for p in PASTAS:
    os.makedirs(os.path.join("/content/drive/MyDrive/PALESTRA-IA", p), exist_ok=True)
print(f"{len(PASTAS)} pastas prontas em {DRIVE}")
for p in PASTAS:
    print("  ", p)

In [ ]:
# ── 3. baixa os modelos base e guarda no Drive ──
#    (baixar agora = na palestra nada depende da internet do local)
import shutil, os
DESTINO = "/content/drive/MyDrive/PALESTRA-IA/00-pesos"
MODELOS = ["yolo11n.pt", "yolo11n-seg.pt", "yolo11n-pose.pt", "yolo11n-cls.pt"]

for m in MODELOS:
    if os.path.exists(f"{DESTINO}/{m}"):
        print("ja tenho:", m); continue
    YOLO(m)                       # baixa para o diretorio corrente
    shutil.copy(m, f"{DESTINO}/{m}")
    print("baixado  :", m)

print("\nconteudo de", DESTINO)
for f in sorted(os.listdir(DESTINO)):
    print("  ", f, round(os.path.getsize(f"{DESTINO}/{f}")/1e6, 1), "MB")

In [ ]:
# ── 4. checklist: o que ainda falta voce colocar ──
import os, glob

def contar(p):
    return len([f for f in glob.glob(os.path.join(DRIVE, p, "*"))
                if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp", ".bmp"))])

ALVOS = [
    ("04-garrafas/inferencia",        5,  "fotos com VARIAS garrafas na cena"),
    ("04-garrafas/treino/train/lacrada", 60, "garrafa lacrada, uma por foto"),
    ("04-garrafas/treino/train/aberta",  60, "garrafa aberta, uma por foto"),
    ("04-garrafas/treino/val/lacrada",   15, "idem, fotos que o treino nao ve"),
    ("04-garrafas/treino/val/aberta",    15, "idem, fotos que o treino nao ve"),
    ("05-epi/treino/train/com_epi",      80, "pessoa com capacete/oculos"),
    ("05-epi/treino/train/sem_epi",      80, "pessoa sem protecao"),
    ("05-epi/treino/val/com_epi",        20, "idem, fotos que o treino nao ve"),
    ("05-epi/treino/val/sem_epi",        20, "idem, fotos que o treino nao ve"),
]

print(f"{'pasta':<38} {'tem':>5} {'meta':>5}   situacao")
print("-" * 78)
tudo_ok = True
for p, meta, desc in ALVOS:
    n = contar(p)
    ok = n >= meta
    tudo_ok &= ok
    print(f"{p:<38} {n:>5} {meta:>5}   {'OK' if ok else 'FALTA — ' + desc}")
print("-" * 78)
print("TUDO PRONTO" if tudo_ok else "ainda faltam imagens — veja as linhas marcadas FALTA")

## Como as pastas viram aprendizado

O YOLO de classificação lê a estrutura de pastas **como se fosse o gabarito**:
o nome da pasta é a resposta certa daquelas imagens. Você não precisa marcar
nada, desenhar caixa nem instalar ferramenta de anotação — basta separar.

```
treino/
├── train/          ← o modelo estuda por aqui
│   ├── lacrada/
│   └── aberta/
└── val/            ← a prova: fotos que ele nunca viu
    ├── lacrada/
    └── aberta/
```

É essa a frase para o palco: **"eu não programei nenhuma regra sobre tampa de
garrafa. Eu só separei as fotos em duas pastas."**